<a href="https://colab.research.google.com/github/Joe-Something/AAI2026/blob/main/Ex3_AISelfReflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# Exercise 3: Self-Reflection Prompt for Improving Output
# Environment: Google Colab
# Library: google-genai
# ==============================================================================

!pip install -q google-genai

import os
import time
from google import genai
from google.genai import errors
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

PRIMARY_MODEL = "gemini-3.6-flash"

def safe_generate_content(prompt):
    for attempt in range(5):
        try:
            return client.models.generate_content(
                model=PRIMARY_MODEL,
                contents=prompt
            )
        except errors.ServerError:
            time.sleep(2 ** attempt)
        except errors.APIError as e:
            if e.code == 429:
                wait_time = 20 + (attempt * 5)
                print(f"[Rate Limit 429] Waiting {wait_time}s before retry (attempt {attempt + 1}/5)...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"API Error ({e.code}): {e.message}")
        except Exception as e:
            if getattr(e, 'code', None) == 429 or "429" in str(e):
                wait_time = 20 + (attempt * 5)
                print(f"[Rate Limit 429] Waiting {wait_time}s before retry (attempt {attempt + 1}/5)...")
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"Unexpected error: {e}")

    raise RuntimeError(f"Failed to generate content with model {PRIMARY_MODEL} after retries.")

source_article = """
Artificial Intelligence and Machine Learning models are increasingly being deployed in cloud-native microservice architectures.
However, inference latency remains a critical bottleneck for real-time applications such as autonomous driving, financial fraud detection, and interactive AI agents.
To mitigate high round-trip latency, edge computing strategies push model deployment directly onto user devices or localized edge nodes.
While edge deployment reduces latency from 150ms down to under 15ms, it introduces severe memory and compute constraints.
Techniques like 4-bit quantization, network pruning, and knowledge distillation have emerged as essential optimization methods to compress parameters without severe accuracy drops.
Organizations adopting a hybrid approach—running quantized lightweight models at the edge for low-latency scoring and offloading complex processing to central cloud clusters—report a 40% reduction in operational infrastructure costs alongside improved reliability during network outages.
"""

print("=== SOURCE TEXT ===")
print(source_article.strip())
print("=" * 60)

# ------------------------------------------------------------------------------
# STEP 1: Initial Generation
# ------------------------------------------------------------------------------
initial_prompt = f"""
Summarize the following article regarding AI edge deployment:

Article:
{source_article}
"""

initial_response = safe_generate_content(initial_prompt)
before_summary = initial_response.text

print("\n--- BEFORE SUMMARY (INITIAL GENERATION) ---")
print(before_summary.strip())

time.sleep(12)  # Pause to keep within 5 RPM ceiling

# ------------------------------------------------------------------------------
# STEP 2: Self-Reflection & Critique Step
# ------------------------------------------------------------------------------
critique_prompt = f"""
You are an executive communications editor. Perform a rigorous critique of the 'Original Summary' against the following strict rubric criteria:

Source Article:
{source_article}

Original Summary:
{before_summary}

Rubric Criteria for Critique:
1. Accuracy & Technical Context: Does it explicitly mention latency metrics (150ms vs 15ms), key compression techniques (quantization, pruning), and business impact (40% cost reduction)?
2. Target Audience & Tone: Is it tailored specifically for an C-suite executive audience (concise, high-level impact, zero jargon filler)?
3. Length & Formatting Constraints: Is the summary UNDER 120 words total AND structured as EXACTLY 3 bullet points?

Provide a point-by-point critique detailing what passed and what failed.
"""

critique_response = safe_generate_content(critique_prompt)
self_critique = critique_response.text

print("\n--- SELF-CRITIQUE OUTPUT ---")
print(self_critique.strip())

time.sleep(12)  # Pause to keep within 5 RPM ceiling

# ------------------------------------------------------------------------------
# STEP 3: Revision Step Based on Critique
# ------------------------------------------------------------------------------
revision_prompt = f"""
You are an expert technical editor. Revise the summary based on the critique provided below.

Original Summary:
{before_summary}

Critique & Rules to Apply:
{self_critique}

Mandatory Output Constraints:
- MUST be formatted as EXACTLY 3 bullet points.
- MUST be under 120 words in total length.
- Tone: Executive-level, authoritative, and direct.
- Must preserve core metrics: latency reduction (150ms to <15ms), compression methods, and 40% cost savings.

Output ONLY the final revised summary.
"""

revision_response = safe_generate_content(revision_prompt)
after_summary = revision_response.text

print("\n--- AFTER SUMMARY (REVISED GENERATION) ---")
print(after_summary.strip())

=== SOURCE TEXT ===
Artificial Intelligence and Machine Learning models are increasingly being deployed in cloud-native microservice architectures. 
However, inference latency remains a critical bottleneck for real-time applications such as autonomous driving, financial fraud detection, and interactive AI agents. 
To mitigate high round-trip latency, edge computing strategies push model deployment directly onto user devices or localized edge nodes. 
While edge deployment reduces latency from 150ms down to under 15ms, it introduces severe memory and compute constraints. 
Techniques like 4-bit quantization, network pruning, and knowledge distillation have emerged as essential optimization methods to compress parameters without severe accuracy drops. 
Organizations adopting a hybrid approach—running quantized lightweight models at the edge for low-latency scoring and offloading complex processing to central cloud clusters—report a 40% reduction in operational infrastructure costs alongsid